### Fine Tuned Resnet on ImageNet Dataset

In [5]:
import os
import torch
import random
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.optim import Adam
from torch.utils.data import DataLoader, random_split
import torchvision.utils as vutils
from torchvision.utils import save_image


In [6]:
path = os.path.join(os.getcwd(), 'data','augmented_animals')
workers = 8
batch_size = 64
image_size = 224
nc = 3
latent_dim = 100
gen_feature = 64
dis_feature = 64
num_epochs = 100
lr = 0.0002
beta1 = 0.5
ngpu = 1
num_classes = 90
device = torch.device("cuda" if (torch.cuda.is_available() and ngpu > 0) else "cpu")


# We can use an image folder dataset the way we have it setup.
# Create the dataset
mean = torch.tensor([0.50869316, 0.50057036, 0.44047505])
std = torch.tensor([0.20191325, 0.19737212, 0.20100889])
dataset = datasets.ImageFolder(root=path,
                           transform=transforms.Compose([
                               transforms.Resize(image_size),
                               transforms.CenterCrop(image_size),
                               transforms.ToTensor(),
                               transforms.Normalize(mean,std),
                           ]))
class_names = dataset.classes
# Create the dataloader
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                                         shuffle=True, num_workers=workers)


In [7]:
train_size=int(0.8*len(dataset))
test_size=len(dataset)-train_size

train_dataset, test_dataset=random_split(dataset, [train_size, test_size])

train_loader=DataLoader(train_dataset, batch_size=64, shuffle=True,num_workers=workers)
test_loader=DataLoader(test_dataset, batch_size=4, shuffle=False)

In [8]:
from torchvision.models import resnet50,ResNet50_Weights

model=resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(device)
model.fc = nn.Linear(2048,num_classes).to(device)
criterion=nn.CrossEntropyLoss()
optimizer=Adam(model.parameters(), lr=lr)

for epoch in range(num_epochs):
    model.train()
    running_loss=0
    for image,labels in train_loader:
        image=image.to(device)
        labels=labels.to(device)

         # Forward pass
        outputs = model(image)
        loss = criterion(outputs, labels)

        # Backpropagation and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')


Epoch [1/100], Loss: 1.0784
Epoch [2/100], Loss: 0.0576
Epoch [3/100], Loss: 0.0199
Epoch [4/100], Loss: 0.0388
Epoch [5/100], Loss: 0.0441
Epoch [6/100], Loss: 0.0274
Epoch [7/100], Loss: 0.0403
Epoch [8/100], Loss: 0.0302
Epoch [9/100], Loss: 0.0389
Epoch [10/100], Loss: 0.0156
Epoch [11/100], Loss: 0.0159
Epoch [12/100], Loss: 0.0163
Epoch [13/100], Loss: 0.0205
Epoch [14/100], Loss: 0.0436
Epoch [15/100], Loss: 0.0225
Epoch [16/100], Loss: 0.0155
Epoch [17/100], Loss: 0.0121
Epoch [18/100], Loss: 0.0152
Epoch [19/100], Loss: 0.0320
Epoch [20/100], Loss: 0.0313
Epoch [21/100], Loss: 0.0174
Epoch [22/100], Loss: 0.0051
Epoch [23/100], Loss: 0.0018
Epoch [24/100], Loss: 0.0008
Epoch [25/100], Loss: 0.0004
Epoch [26/100], Loss: 0.0005
Epoch [27/100], Loss: 0.0290
Epoch [28/100], Loss: 0.0673
Epoch [29/100], Loss: 0.0236
Epoch [30/100], Loss: 0.0205
Epoch [31/100], Loss: 0.0196
Epoch [32/100], Loss: 0.0053
Epoch [33/100], Loss: 0.0049
Epoch [34/100], Loss: 0.0031
Epoch [35/100], Loss: 0

In [9]:
def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print(f'Accuracy: {100 * correct / total:.2f}%')
    # Calculate the F1 score using PyTorch
    all_preds = torch.tensor(all_preds)
    all_labels = torch.tensor(all_labels)

    # Initialize counters for TP, FP, FN
    TP = torch.zeros(num_classes)
    FP = torch.zeros(num_classes)
    FN = torch.zeros(num_classes)

    for i in range(num_classes):
        TP[i] = ((all_preds == i) & (all_labels == i)).sum().item()
        FP[i] = ((all_preds == i) & (all_labels != i)).sum().item()
        FN[i] = ((all_preds != i) & (all_labels == i)).sum().item()

    # Calculate precision, recall, and F1-score for each class
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    print(f'Precision: {precision.mean().item():.2f}')
    print(f'Recall: {recall.mean().item():.2f}')
    print(f'F1 Score: {f1.mean().item():.2f}')
    return None

# Assuming `val_loader` is defined similarly to `train_loader`
val_loader=test_loader
evaluate(model, val_loader)


Accuracy: 97.84%
Precision: 0.98
Recall: 0.98
F1 Score: 0.98


### Comparision of ResNet Pretrained  with MLP Trained with Decoded latents

| Model Name       | Accuracy (%) | F1 Score   |
|------------------|--------------|------------|
| Resnet-50         | 97.84         | 0.98      |
| MLP trained on Decoded latents       | 31.03        | 0.309      |
#### Observation:
As we can see, the Resnet50 model gives us much higher accuracy than in case of the MLP. \
This shows that simple MLP layers are not so good learners of representaion of high dimensional data. 